In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# 1. knowledge base 구성을 위한 데이터 생성

In [2]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax/소득세법_with_table.docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,   # 문서를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n", "\n", " ", ""]
)

documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

<timed exec>:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.


CPU times: total: 3.39 s
Wall time: 3.81 s


180

In [3]:
documents[43]

Document(metadata={'source': 'tax/소득세법_with_table.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1,400만원 이하\n\n과세표준의 6퍼센트\n\n1,400만원 초과 5,000만원 이하\n\n84만원 + (1,400만원을 초과하는 금액의 15퍼센트)\n\n5,000만원 초과 8,800만원 이하\n\n624만원 + (5,000만원을 초과하는 금액의 24퍼센트)\n\n8,800만원 초과 1억 5천만원 이하\n\n1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)\n\n1억5천만원 초과 3억원 이하\n\n3,706만원 + (1억5천만원을 초과하는 금액의 38퍼센

In [4]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv(override=True)
embedding = UpstageEmbeddings(
    model = 'embedding-query'
)

In [5]:
len(embedding.embed_query("소득세법"))

4096

In [17]:
import time
from pinecone import Pinecone, ServerlessSpec

# pinecone 객체 생성
pc = Pinecone()

# index 설정
index_name = 'tax-index-table'
dimension = len(embedding.embed_query("소득세법"))

# 인덱스 생성 (없을 경우에만 생성)
existing_indexes = [index.name for index in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)
    print(f'인덱스 생성 : {index_name}')
else:
    print(f'중복된 인덱스 : {index_name}')

인덱스 생성 : tax-index-table


In [18]:
# 인덱스 정보 확인
print([index.name for index in pc.list_indexes()])
print(pc.describe_index(index_name).status['ready'])
index = pc.Index(index_name)
print(index.describe_index_stats())

['tax-index-openai', 'tax-index', 'tax-index-table']
True
{'dimension': 4096,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [19]:
# 문서 임베딩 및 클라우드 DB에 저장
from langchain_pinecone import PineconeVectorStore
database = PineconeVectorStore.from_documents(
    documents=documents,
    embedding=embedding,
    index_name=index_name
)

In [10]:
# 클라우드 DB의 vectorDB load
from langchain_pinecone import PineconeVectorStore
index_name = 'tax-index-table'
database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

# 2. 답변 생성 전 Retrieval 확인

In [20]:
query = "연봉이 5천만원 직장인의 소득세는 얼마인가요?"
retrieval_docs = database.similarity_search(query, k=3)

In [21]:
retriever = database.as_retriever(
    search_type='similarity',
    search_kwargs={'k':2}
    )

retrieval_docs = retriever.invoke('연봉이 5천만원인 직장인의 소득세는 얼마인가요?')

In [22]:
retrieval_doc = "\n--\n".join([doc.page_content for doc in retrieval_docs])

In [23]:
'55조' in retrieval_doc

True

# 3. 답변 생성

In [24]:
# from openai import OpenAI
# client = OpenAI()

# query = "연봉이 5천만원 직장인의 소득세는 얼마인가요?"

# response = client.chat.completions.create(
#     model='gpt-4o-mini',
#     messages=[
#         {'role':'system', 'content':'당신은 최고의 한국 소득세법 전문가입니다'},
#         {'role':'user',
#          'content':f'''-[context]를 참고하여 사용자의 질문에 10줄 이내로 답변하세요
#          - [context] : {retrieval_doc},
#          - 질문 : {query}'''
#         }
#     ],
#     temperature=0.2
# )

In [ ]:
result = response.choices[0].message.content

In [25]:
from langchain_upstage import ChatUpstage

llm = ChatUpstage(
    model = 'solar-pro4',
    reasoning_effort='low'
)

In [26]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가 입니다
-[context]를 참고하여 사용자의 질문에 10줄 이내로 답변하세요
- [context] : {retrieval_doc},
- 질문 : {query}"""

ai_msg = llm.invoke(prompt)

In [27]:
print(ai_msg.content)

제공된 소득세법 조항에 근거한 계산 예시입니다. 실제 세금은 연봉에서 공제액을 뺀 **종합소득과세표준**을 기준으로 하므로 실제 금액은 다를 수 있습니다.

1. **산출세액 (제55조)**: 과세표준 5,000만원 시 '1,400만원 초과 5,000만원 이하' 구간 적용.
→ 84만원 + (3,600만원 × 15%) = **624만원**.
2. **근로소득세액공제 (제59조)**: 총급여 5,000만원 (3,300~7,000만원 구간) 적용.
→ 74만원 - (1,700만원 × 0.008) = 60.4만원 (최소한도 66만원 적용).
3. **결정세액**: 산출세액 624만원 - 세액공제 66만원 = **558만원** (지방소득세 별도).

※ 제공된 문맥에는 근로소득공제 등 과세표준 결정 공제 항목이 없어, 연봉 5,000만원을 과세표준으로 가정한 계산입니다. 실제로는 과세표준이 낮아져 세금이 더 적을 수 있습니다.


# LangChian

In [28]:
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

# 임베딩 & LLM 초기화
load_dotenv()

embedding = UpstageEmbeddings(model='embedding-query')
llm = ChatUpstage(model='solar-pro4')

# vector DB load
# pc = Pinecone()
index_name = 'tax-index-table'
vectorStore = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

# Retriever 생성
retriever = vectorStore.as_retriever(
    search_type='similarity', 
    search_kwargs={'k':2}
)

# 프롬프트 템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다 다음 문맥을 참고하여 질문에 답하세요 답을 모르면 모른다고 답하세요 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 :"""
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])

In [29]:
# RAG 렝체인 구성 (LCEL방식)
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    | prompt
    | llm
    | StrOutputParser()
)
result = rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')
print(result)

제공된 문맥에는 근로소득공제 등 소득공제 항목이 명시되어 있지 않아 정확한 소득세를 산출할 수 없습니다. 다만 과세표준이 연봉 5,000만원과 동일하다고 가정하더라도 직장인은 근로소득공제와 각종 세액공제를 적용받으므로 실제 납부세액은 달라집니다. 정확한 세액은 소득·세액공제 내역을 반영한 종합소득세 계산을 통해 확인해야 하므로 이 답변만으로는 알 수 없습니다.
